In [1]:
!pip install -q sentence-transformers

In [2]:
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer, util

In [3]:
TOPICS = [
    "food", "family", "holiday", "travel", "health", "fitness", "education",
    "technology", "music", "movies", "books", "fashion", "nature", "pets",
    "business", "art", "science", "gaming", "home", "garden",
]

In [4]:
en_word_freq = pd.read_csv("en_word_freq.csv")
print(f"Total English words: {len(en_word_freq)}")

Total English words: 313282


In [5]:
words_list = en_word_freq["word"].astype(str).tolist()

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)

Using device: cuda


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [6]:
# =========================================================
# Embed topics once
# =========================================================
topic_embeddings = model.encode(TOPICS, normalize_embeddings=True, show_progress_bar=True)

# =========================================================
# Embed ALL English words
# =========================================================
word_embeddings = model.encode(
    words_list,
    normalize_embeddings=True,
    batch_size=256,
    show_progress_bar=True,
)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1224 [00:00<?, ?it/s]

In [7]:
# =========================================================
# Cosine similarity: every word x every topic
# =========================================================
sim_matrix = util.cos_sim(word_embeddings, topic_embeddings).numpy()

english_topic_sim_df = pd.DataFrame(sim_matrix, columns=TOPICS)
english_topic_sim_df.insert(0, "word", words_list)
english_topic_sim_df = english_topic_sim_df.merge(
    en_word_freq[["word", "freq", "rank"]], on="word", how="left"
)

print(english_topic_sim_df.shape)
english_topic_sim_df.head()

(313282, 23)


,word,food,family,holiday,travel,health,fitness,education,technology,music,...,nature,pets,business,art,science,gaming,home,garden,freq,rank
0,the,0.293102,0.398883,0.314613,0.260077,0.303626,0.236690,0.339314,0.312440,0.325836,...,0.242575,0.339650,0.389597,0.334970,0.279610,0.286165,0.372626,0.276455,1.259718e+11,1.0
1,of,0.274326,0.272821,0.211824,0.222237,0.290212,0.197117,0.337648,0.217173,0.283157,...,0.194011,0.209818,0.282278,0.379522,0.263500,0.269427,0.290604,0.193574,7.746622e+10,2.0
2,and,0.239577,0.267868,0.232922,0.223515,0.206200,0.160918,0.264099,0.226154,0.246761,...,0.152153,0.172069,0.275493,0.241066,0.219241,0.221297,0.227532,0.180739,5.560701e+10,3.0
3,to,0.356763,0.359206,0.290034,0.381080,0.271793,0.244982,0.391741,0.277506,0.324421,...,0.215108,0.217630,0.358169,0.322486,0.327380,0.321428,0.407724,0.241095,4.662282e+10,4.0
4,in,0.314291,0.347955,0.271072,0.273122,0.264350,0.242547,0.382426,0.267217,0.296596,...,0.277333,0.241922,0.410167,0.301462,0.332439,0.330786,0.441561,0.268029,4.006948e+10,5.0


In [8]:
# =========================================================
# Save as a reusable dataset
# =========================================================
english_topic_sim_df.to_csv("english_word_topic_similarity.csv", index=False)